# FockPARF Improvement Sweep — Bridging the PPL Gap to Attention

## Motivation

FockPARF with $V_\theta$ regularisation ($\lambda_V = 1$) achieves **190.2 PPL** on
TinyShakespeare — 15.5 PPL better than unregularised, but still **~40 PPL behind**
the attention baseline (~150 PPL) and **~57 PPL behind** Hybrid SPLM+Attn (133 PPL).

This notebook tests five strategies to close that gap, plus three
γ/v_hidden confound-resolution cells (G1–G3) added after the PARF vreg sweep
revealed that prior results used γ_init=0.15 and v_hidden=128 rather than
the SPLM em_ln optimum γ*=0.10 and v_hidden=512.

## Cell structure

| Cell | Strategy | Key changes vs FR4 baseline |
|------|----------|-----------------------------|
| `P1` | Hybrid FockPARF+Attn (k=4, m=4) | 4 attention blocks + 4 FockPARF layers |
| `P2` | Scaled FockPARF (v_hidden=512, 8000 steps) | 4× wider V_θ, 2× longer training, λ_V=1 |
| `P3` | More registers + score entropy reg | M=32, entropy reg on Gumbel scores |
| `P4` | Width scaling (d=256) | Double embedding dim, L=8, λ_V=1 |
| `P5` | Phased gate training | Freeze gates for first 1000 steps |
| `G1` | PARF γ sweep at λ_V=10⁻⁴ | fixed_γ=0.10, v_hidden=128 (match em_ln γ*) |
| `G2` | PARF v_hidden=512 at λ_V=10⁻⁴ | v_hidden=512, γ_init=0.10 (match em_ln capacity) |
| `G3` | SPLM em_ln vreg baseline | v_hidden=512, fixed_γ=0.10, λ_V=10⁻⁴ (no V_φ) |

Run the notebook once per `CELL`; outputs persist on GDrive across sessions.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'P1'       # one of: 'P1'..'P5' | 'G1' | 'G2' | 'G3'
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula'
GDRIVE_OUT_REL  = 'semsimula_vreg'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is.')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    RESULTS_ROOT = GDRIVE_OUT / 'fockparf_improvement'
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow scikit-learn')

else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError('Could not locate the semsimula repo root.')
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / 'fockparf_improvement'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

CONS_ARCH_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
SARF_DIR      = CONS_ARCH_DIR / 'sarf_mass_variant'
PARF_DIR      = CONS_ARCH_DIR / 'parf'
ATTRACTOR_DIR = CONS_ARCH_DIR / 'attractor_analysis'
for p in (str(REPO_ROOT), str(CONS_ARCH_DIR), str(SARF_DIR),
          str(PARF_DIR), str(ATTRACTOR_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 1. Disable TF32, set seeds, pick device

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F_torch

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device')
else:
    device = 'cpu'
    print('CPU only — runs will be slow.')
print(f'device = {device}')

## 2. Translate `CELL` switch into config

Each cell defines a different architectural or training strategy.
All cells use $\lambda_V = 1$ (from the V_θ regularisation sweep finding
that regularisation helps FockPARF) unless otherwise noted.

In [ ]:
IMPROVEMENT_RECIPES = {
    'P1': {
        'desc': 'Hybrid FockPARF+Attn (k=4, m=4)',
        'model': 'hybrid_fock_parf',
        'd': 128, 'L': 4, 'v_hidden': 128, 'v_depth': 3,
        'n_attn': 4, 'n_head': 4, 'mlp_mult': 4,
        'n_registers': 16, 'creation_gate_hidden': 32,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1.0, 'steps': 4000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': False,
    },
    'P2': {
        'desc': 'Scaled FockPARF: v_hidden=512, 8000 steps',
        'model': 'fock_parf',
        'd': 128, 'L': 8, 'v_hidden': 512, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 16, 'creation_gate_hidden': 32,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1.0, 'steps': 8000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': False,
    },
    'P3': {
        'desc': 'More registers (M=32) + score entropy reg',
        'model': 'fock_parf',
        'd': 128, 'L': 8, 'v_hidden': 128, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 32, 'creation_gate_hidden': 64,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1.0, 'steps': 4000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.01,
        'phase_gates': False,
    },
    'P4': {
        'desc': 'Width scaling: d=256, full rescale',
        'model': 'fock_parf',
        'd': 256, 'L': 8, 'v_hidden': 256, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 32, 'creation_gate_hidden': 64,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1.0, 'steps': 4000,
        'batch': 8, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': False,
    },
    'P5': {
        'desc': 'Phased gate training (freeze gates 1000 steps)',
        'model': 'fock_parf',
        'd': 128, 'L': 8, 'v_hidden': 128, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 16, 'creation_gate_hidden': 32,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1.0, 'steps': 4000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': True,
        'gate_freeze_steps': 1000,
    },
    # --- G-series: gamma / v_hidden / em_ln confound resolution ---
    'G1': {
        'desc': 'PARF gamma=0.10 (fixed) at lambda_V=1e-4',
        'model': 'parf',
        'd': 128, 'L': 8, 'v_hidden': 128, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 0, 'creation_gate_hidden': 0,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1e-4, 'steps': 4000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': False,
        'init_gamma': 0.10, 'fixed_gamma': 0.10,
    },
    'G2': {
        'desc': 'PARF v_hidden=512, gamma_init=0.10 at lambda_V=1e-4',
        'model': 'parf',
        'd': 128, 'L': 8, 'v_hidden': 512, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 0, 'creation_gate_hidden': 0,
        'top_k': 16, 'score_head_hidden': 32,
        'lambda_v': 1e-4, 'steps': 4000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': False,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
    'G3': {
        'desc': 'SPLM em_ln (v_hidden=512, gamma=0.10) at lambda_V=1e-4',
        'model': 'splm',
        'd': 128, 'L': 8, 'v_hidden': 512, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 0, 'creation_gate_hidden': 0,
        'top_k': 0, 'score_head_hidden': 0,
        'lambda_v': 1e-4, 'steps': 4000,
        'batch': 16, 'block': 128,
        'score_entropy_reg': 0.0,
        'phase_gates': False,
        'init_gamma': 0.10, 'fixed_gamma': 0.10,
    },
}
if CELL not in IMPROVEMENT_RECIPES:
    raise ValueError(f'CELL must be one of {list(IMPROVEMENT_RECIPES)}; got {CELL!r}')

recipe = IMPROVEMENT_RECIPES[CELL]
LAMBDA_V       = recipe['lambda_v']
STEPS          = recipe['steps']
D              = recipe['d']
L              = recipe['L']
V_HIDDEN       = recipe['v_hidden']
V_DEPTH        = recipe['v_depth']
N_ATTN         = recipe['n_attn']
N_HEAD         = recipe['n_head']
MLP_MULT       = recipe['mlp_mult']
N_REGISTERS    = recipe['n_registers']
CREATION_GATE_HIDDEN = recipe['creation_gate_hidden']
TOP_K          = recipe['top_k']
SCORE_HEAD_HIDDEN = recipe['score_head_hidden']
BATCH          = recipe['batch']
BLOCK          = recipe['block']
SCORE_ENTROPY_REG = recipe['score_entropy_reg']
PHASE_GATES    = recipe['phase_gates']
GATE_FREEZE_STEPS = recipe.get('gate_freeze_steps', 0)
MODEL_KIND     = recipe['model']
INIT_GAMMA     = recipe.get('init_gamma', 0.15)
FIXED_GAMMA    = recipe.get('fixed_gamma', None)

VOCAB_SIZE  = 50257
MAX_LEN     = 256
DT          = 1.0
V_PHI_KIND  = 'structural'
V_PHI_D_TYPE = 16
V_PHI_D_ANGLE = 8
V_PHI_PHI_HIDDEN = 32
V_PHI_THETA_HIDDEN = 32
V_PHI_MLP_HIDDEN = 64
GUMBEL_TAU_INIT = 1.0
GUMBEL_TAU_MIN = 0.1
LR = 5e-4
WD = 0.01
WARMUP = 200
GRAD_CLIP = 1.0
EVAL_INTERVAL = 200
EVAL_ITERS = 40
LOG_INTERVAL = 50
STACK_DISCIPLINE = True

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  model={MODEL_KIND}  d={D}  L={L}  v_hidden={V_HIDDEN}  M={N_REGISTERS}')
print(f'  n_attn={N_ATTN}  lambda_V={LAMBDA_V}  steps={STEPS}  batch={BATCH}')
print(f'  score_entropy_reg={SCORE_ENTROPY_REG}  phase_gates={PHASE_GATES}')
print(f'  init_gamma={INIT_GAMMA}  fixed_gamma={FIXED_GAMMA}')

## 3. Load TinyShakespeare

In [ ]:
from data_module import load_tiny_shakespeare, get_batch

train_ids, val_ids = load_tiny_shakespeare()
print(f'tokens: train={len(train_ids):,}  val={len(val_ids):,}')

## 4. Build model (FockPARF or Hybrid FockPARF+Attn)

In [ ]:
from parf.model_fock_parf import FockPARFLM, FockPARFConfig
from parf.model_hybrid_fock_parf import HybridFockPARF, HybridFockPARFConfig
from parf.model_parf_sparse import SparsePARFLM, SparsePARFConfig
from sarf_mass_variant.model_sarf_mass import causal_cumulative_mean
from energetic_minima.model_ln import (
    ScalarPotentialLMSARFMassLN, SPLMSARFMassLNConfig,
)

# Locate or build logfreq surprisal
BUNDLED_LOGFREQ = SARF_DIR / 'results' / 'logfreq_surprisal.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_shakespeare.npy'

if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled logfreq surprisal: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq surprisal: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

torch.manual_seed(SEED)

gamma_kwargs = dict(init_gamma=INIT_GAMMA)
if FIXED_GAMMA is not None:
    gamma_kwargs['fixed_gamma'] = FIXED_GAMMA

if MODEL_KIND == 'hybrid_fock_parf':
    cfg = HybridFockPARFConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
        L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
        v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_PATH),
        top_k=TOP_K,
        score_head_hidden=SCORE_HEAD_HIDDEN,
        gumbel_tau_init=GUMBEL_TAU_INIT,
        gumbel_tau_min=GUMBEL_TAU_MIN,
        n_registers=N_REGISTERS,
        creation_gate_hidden=CREATION_GATE_HIDDEN,
        stack_discipline=STACK_DISCIPLINE,
        n_attn=N_ATTN,
        n_head=N_HEAD,
        mlp_mult=MLP_MULT,
        **gamma_kwargs,
    )
    model = HybridFockPARF(cfg).to(device)
    n_attn_oh = model.get_attn_overhead()
    print(f'Hybrid FockPARF: n_attn={N_ATTN}, L={L} FockPARF layers')
    print(f'  attn_overhead={n_attn_oh:,}')
elif MODEL_KIND == 'splm':
    cfg = SPLMSARFMassLNConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
        L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_PATH),
        ln_after_step=True,
        causal_force=True,
        **gamma_kwargs,
    )
    model = ScalarPotentialLMSARFMassLN(cfg).to(device)
    print(f'SPLM em_ln: L={L}, v_hidden={V_HIDDEN}, gamma={INIT_GAMMA}/{FIXED_GAMMA}')
elif MODEL_KIND == 'parf':
    cfg = SparsePARFConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
        L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
        v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_PATH),
        top_k=TOP_K,
        score_head_hidden=SCORE_HEAD_HIDDEN,
        gumbel_tau_init=GUMBEL_TAU_INIT,
        gumbel_tau_min=GUMBEL_TAU_MIN,
        **gamma_kwargs,
    )
    model = SparsePARFLM(cfg).to(device)
    print(f'SparsePARFLM: L={L}, v_hidden={V_HIDDEN}, gamma={INIT_GAMMA}/{FIXED_GAMMA}')
else:
    cfg = FockPARFConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
        L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
        v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_PATH),
        top_k=TOP_K,
        score_head_hidden=SCORE_HEAD_HIDDEN,
        gumbel_tau_init=GUMBEL_TAU_INIT,
        gumbel_tau_min=GUMBEL_TAU_MIN,
        n_registers=N_REGISTERS,
        creation_gate_hidden=CREATION_GATE_HIDDEN,
        stack_discipline=STACK_DISCIPLINE,
        **gamma_kwargs,
    )
    model = FockPARFLM(cfg).to(device)
    print(f'FockPARF: L={L}, M={N_REGISTERS}')

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters()) if hasattr(model, 'V_phi') else 0
n_fock = model.get_register_overhead() if hasattr(model, 'get_register_overhead') else 0
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  V_phi={n_v_phi:,}  '
      f'Fock_overhead={n_fock:,}')

## 5. Training loop

Features beyond the standard FockPARF training loop:
- **V_θ regularisation** ($\lambda_V$) applied via decomposed forward
- **Score entropy regularisation** (P3): penalises low-entropy Gumbel score distributions
- **Phased gate training** (P5): freezes creation/destruction gates for the first N steps
- **Gumbel τ annealing** as in the standard FockPARF trainer

In [ ]:
import math, time, json


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def tau_at(step):
    anneal_fraction = 0.8
    warm = int((1.0 - anneal_fraction) * STEPS)
    if step < warm:
        return GUMBEL_TAU_INIT
    if step >= STEPS:
        return GUMBEL_TAU_MIN
    progress = (step - warm) / max(STEPS - warm, 1)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * min(progress, 1.0)


def forward_with_vreg(model, x, targets, lambda_v):
    """Decomposed forward for FockPARF, HybridFockPARF, PARF, and SPLM."""
    h0 = model._embed(x)

    if MODEL_KIND == 'hybrid_fock_parf':
        h_attn, _ = model._attn_stack(h0)
        h_k = model.ln_boundary(h_attn)
        h_L, _ = model._stack_forward(h_k, x, return_trajectory=False)
    elif MODEL_KIND == 'splm':
        h_L, _, _ = model.integrate(x, h0, return_trajectory=False)
    else:
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, model.cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


def compute_score_entropy(model, x):
    """Compute mean entropy of the Gumbel-softmax score distribution.

    Low entropy means the score head collapsed to a degenerate routing.
    We penalise low entropy to encourage diverse pair selection.
    """
    h0 = model._embed(x)
    if MODEL_KIND == 'hybrid_fock_parf':
        h_attn, _ = model._attn_stack(h0)
        h = model.ln_boundary(h_attn)
    else:
        h = h0
    B, T, d = h.shape
    if not hasattr(model, 'score_head'):
        return torch.tensor(0.0, device=x.device)
    scores = model.score_head(h, h.detach())
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=-1)
    scores = scores.masked_fill(~mask.unsqueeze(0), -1e9)
    probs = torch.softmax(scores / max(model.gumbel_tau, 0.01), dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -(probs * log_probs).sum(dim=-1).mean()
    return entropy


def freeze_gates(model, freeze=True):
    for gate in model.creation_gates:
        for p in gate.parameters():
            p.requires_grad = not freeze
    for gate in model.destruction_gates:
        for p in gate.parameters():
            p.requires_grad = not freeze
    status = 'FROZEN' if freeze else 'UNFROZEN'
    print(f'  [gate control] creation+destruction gates: {status}')


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


# Optionally freeze gates for phased training (P5) — only applies to FockPARF models
HAS_GATES = hasattr(model, 'creation_gates')
HAS_GUMBEL = hasattr(model, 'set_gumbel_tau')
if PHASE_GATES and HAS_GATES:
    freeze_gates(model, freeze=True)

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
best_ppl = float('inf')
t0 = time.time()

for step in range(STEPS):
    # Phased gate unfreezing (P5)
    if PHASE_GATES and HAS_GATES and step == GATE_FREEZE_STEPS:
        freeze_gates(model, freeze=False)
        # Rebuild optimiser to include the newly unfrozen parameters
        opt = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr_at(step), betas=(0.9, 0.95), weight_decay=WD,
        )
        print(f'  [phase 2] optimiser rebuilt with all params at step {step}')

    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    if HAS_GUMBEL:
        model.set_gumbel_tau(tau_at(step))

    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)

    # Score entropy regularisation (P3)
    entropy_loss = torch.tensor(0.0, device=device)
    if SCORE_ENTROPY_REG > 0:
        ent = compute_score_entropy(model, x)
        entropy_loss = -SCORE_ENTROPY_REG * ent
        loss = loss + entropy_loss

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        tau_str = f'tau={tau_at(step):.3f}  ' if HAS_GUMBEL else ''
        msg = (f'[{CELL}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'{tau_str}'
               f'ntp={loss_ntp.item():.4f}  '
               f'v_reg={v_reg.item():.4f}  '
               f'total={loss.item():.4f}  '
               f'wall={time.time() - t0:.0f}s')
        if SCORE_ENTROPY_REG > 0:
            msg += f'  ent_loss={entropy_loss.item():.4f}'
        print(msg)

    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        if val_ppl < best_ppl:
            best_ppl = val_ppl
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best_ppl={best_ppl:.2f}')
        log.append({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_ppl,
            'train_loss_ntp': loss_ntp.item(),
            'v_reg': v_reg.item(), 'train_loss_total': loss.item(),
            'lambda_v': LAMBDA_V, 'tau': tau_at(step) if HAS_GUMBEL else 0.0,
        })

print(f'\n[{CELL}] Training done.  total wall = {time.time() - t0:.0f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}  '
      f'best_ppl = {best_ppl:.2f}')

## 6. Save checkpoint and training log

In [ ]:
import dataclasses

RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
full_tag = f'fockparf_improve_{CELL}_{MODEL_KIND}_d{D}_L{L}_M{N_REGISTERS}_seed{SEED}'
ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_cfg': dataclasses.asdict(cfg),
        'variant': MODEL_KIND,
        'cell': CELL,
        'lambda_v': LAMBDA_V,
        'step': STEPS,
        'final_val_ppl': log[-1]['val_ppl'],
        'best_val_ppl': best_ppl,
        'seed': SEED,
        'recipe': recipe,
    },
    ckpt_path,
)
with open(log_path, 'w') as f:
    for row in log:
        f.write(json.dumps(row) + '\n')
print(f'wrote ckpt: {ckpt_path}')
print(f'wrote log : {log_path}')

## 7. Plot val PPL trajectory

In [ ]:
import matplotlib.pyplot as plt

steps_log = [r['step'] for r in log]
ppl_log = [r['val_ppl'] for r in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(steps_log, ppl_log, marker='o', color='#3a6ea5',
        label=f'{CELL}: {recipe["desc"]}')
ax.axhline(190.2, color='gray', linestyle=':', alpha=0.6, label='FR4 baseline (190.2)')
ax.axhline(149.8, color='green', linestyle='--', alpha=0.6, label='Attention baseline (~150)')
ax.axhline(133.0, color='purple', linestyle='--', alpha=0.6, label='Hybrid SPLM best (133)')
ax.set_xlabel('train step')
ax.set_ylabel('val PPL')
ax.set_title(f'{CELL} val PPL  (best = {best_ppl:.2f})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
vreg_log = [r['v_reg'] for r in log]
ax.plot(steps_log, vreg_log, marker='s', color='#b03030', label='mean V_\u03b8\u00b2')
ax.set_xlabel('train step')
ax.set_ylabel('V_\u03b8\u00b2')
ax.set_title(f'{CELL} V_\u03b8 regularisation  (\u03bb_V = {LAMBDA_V})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_training.png', dpi=120)
plt.show()

## 8. Post-training attractor extraction

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    from transformers import GPT2Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
except Exception:
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    class _Tok:
        def encode(self, s): return enc.encode(s)
        def decode(self, ids): return enc.decode(ids)
    tokenizer = _Tok()

PROMPTS = [
    ('narrative',   'The old king sat on the'),
    ('mathematics', 'The theorem states that for every'),
    ('scientific',  'Photosynthesis converts carbon dioxide and'),
    ('dialogue',    'She whispered: I love'),
    ('code',        'def fibonacci(n): return 1 if n < 2 else'),
]

N_SEEDS = 384
DESCENT_STEPS = 1500
DESCENT_LR = 0.05

model.eval()


def build_seeds_simple(h_mean, h_std, n=384):
    gen = torch.Generator(device='cpu').manual_seed(SEED)
    noise = torch.randn(n, D, generator=gen).to(device)
    return h_mean + noise * h_std


def descend_pure(model, xi, seeds, steps=1500, lr=0.05):
    h = seeds.clone().detach().requires_grad_(True)
    xi_b = xi.unsqueeze(0).expand(h.shape[0], -1)
    opt_d = torch.optim.Adam([h], lr=lr)
    for s in range(steps):
        V = model.V_theta(xi_b, h).sum()
        opt_d.zero_grad()
        V.backward()
        opt_d.step()
        if (s + 1) % max(steps // 5, 1) == 0:
            with torch.no_grad():
                v_mean = V.item() / h.shape[0]
            print(f'  [gd] step {s+1:5d}/{steps}  <V>={v_mean:+.3f}')
    h_f = h.detach().clone().requires_grad_(True)
    V = model.V_theta(xi_b, h_f).sum()
    g, = torch.autograd.grad(V, h_f)
    grad_norm = g.norm(dim=-1).detach().cpu().numpy()
    converged = int((grad_norm < 0.05).sum())
    V_final = model.V_theta(xi_b, h_f.detach()).detach().cpu().numpy().ravel()
    return h.detach().cpu().numpy(), grad_norm, V_final, converged


def descend_anchored(model, xi, seeds, h_center, h_std, lam=0.1,
                     steps=1500, lr=0.05):
    h = seeds.clone().detach().requires_grad_(True)
    xi_b = xi.unsqueeze(0).expand(h.shape[0], -1)
    h_c = h_center.to(device)
    h_s = h_std.to(device)
    opt_d = torch.optim.Adam([h], lr=lr)
    for s in range(steps):
        V = model.V_theta(xi_b, h).sum()
        reg = 0.5 * lam * (((h - h_c) / h_s) ** 2).sum()
        loss_a = V + reg
        opt_d.zero_grad()
        loss_a.backward()
        opt_d.step()
    h_f = h.detach().clone().requires_grad_(True)
    V = model.V_theta(xi_b, h_f).sum()
    reg = 0.5 * lam * (((h_f - h_c) / h_s) ** 2).sum()
    g, = torch.autograd.grad(V + reg, h_f)
    grad_norm = g.norm(dim=-1).detach().cpu().numpy()
    converged = int((grad_norm < 0.05).sum())
    V_final = model.V_theta(xi_b, h_f.detach()).detach().cpu().numpy().ravel()
    return h.detach().cpu().numpy(), grad_norm, V_final, converged


def simulate_damped_simple(model, xi, seeds, n_steps):
    h = seeds.clone().detach().to(device)
    v = torch.zeros_like(h)
    xi_b = xi.unsqueeze(0).expand(h.shape[0], -1)
    m_val = float(model.m_global.item())
    gamma_val = float(model.gamma.item())
    dt_val = float(cfg.dt)
    for s in range(n_steps):
        h.requires_grad_(True)
        V = model.V_theta(xi_b, h).sum()
        g, = torch.autograd.grad(V, h)
        h = h.detach()
        f = -g.detach()
        v = (v + dt_val * f / m_val) / (1.0 + dt_val * gamma_val)
        h = h + dt_val * v
    return h.detach().cpu().numpy()


def sweep_clusters(h, K_min=2, K_max=10):
    sils = {}
    best_K, best_sil = K_min, -np.inf
    for K in range(K_min, K_max + 1):
        if len(h) <= K:
            continue
        km = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(h)
        try:
            sil = silhouette_score(h, km.labels_)
        except Exception:
            sil = -np.inf
        sils[K] = float(sil)
        if sil > best_sil:
            best_sil, best_K = sil, K
    km = KMeans(n_clusters=best_K, random_state=SEED, n_init=10).fit(h)
    return best_K, km.labels_, km.cluster_centers_, sils


def decode_centroids(centroids):
    c_t = torch.from_numpy(centroids).to(device).float()
    logits = c_t @ model.E.weight.T
    probs = torch.softmax(logits, dim=-1)
    results = []
    for i in range(len(centroids)):
        top_vals, top_ids = probs[i].topk(5)
        tokens = [(tokenizer.decode([tid.item()]), float(tv.item()))
                  for tid, tv in zip(top_ids, top_vals)]
        results.append(tokens)
    return results

In [ ]:
h_all = []
for _, prompt in PROMPTS:
    ids = torch.tensor(tokenizer.encode(prompt),
                       device=device, dtype=torch.long).unsqueeze(0)
    with torch.enable_grad():
        out = model(ids, return_trajectory=True)
    h_all.append(out[2][-1][0].to(device))
H_all = torch.cat(h_all, dim=0)
h_mean = H_all.mean(0)
h_std = H_all.std(0).clamp_min(1e-3)

attractor_results = {}

for prompt_name, prompt_text in PROMPTS:
    print(f'\n{"="*60}')
    print(f'Prompt: "{prompt_text}" ({prompt_name})')
    print(f'{"="*60}')

    ids = torch.tensor(tokenizer.encode(prompt_text),
                       device=device, dtype=torch.long).unsqueeze(0)
    with torch.enable_grad():
        out = model(ids, return_trajectory=True)
    xi = causal_cumulative_mean(out[2][-1][0:1].to(device))[0, -1, :].detach()

    seeds = build_seeds_simple(h_mean, h_std, n=N_SEEDS)

    print('\n--- Protocol 1: Pure V_\u03b8 descent ---')
    h_gd, gnorm_gd, V_gd, n_conv_gd = descend_pure(
        model, xi, seeds, steps=DESCENT_STEPS, lr=DESCENT_LR)
    print(f'  converged (||\u2207V|| < 0.05): {n_conv_gd}/{N_SEEDS}')
    print(f'  <V> = {V_gd.mean():.1f}  ||h|| = {np.linalg.norm(h_gd, axis=-1).mean():.1f}')
    K_gd, labels_gd, centers_gd, sils_gd = sweep_clusters(h_gd)
    dec_gd = decode_centroids(centers_gd)
    print(f'  K* = {K_gd}  silhouettes = {sils_gd}')
    for i, tokens in enumerate(dec_gd):
        size = (labels_gd == i).sum()
        top3 = ', '.join(f'{repr(t)}: {p:.2f}' for t, p in tokens[:3])
        print(f'    A{i} ({size}): {top3}')

    print('\n--- Protocol 2: Anchored descent (\u03bb_anchor=0.1) ---')
    h_anch, gnorm_anch, V_anch, n_conv_anch = descend_anchored(
        model, xi, seeds, h_mean, h_std, lam=0.1,
        steps=DESCENT_STEPS, lr=DESCENT_LR)
    K_anch, labels_anch, centers_anch, sils_anch = sweep_clusters(h_anch)
    dec_anch = decode_centroids(centers_anch)
    print(f'  K* = {K_anch}  silhouettes = {sils_anch}')
    for i, tokens in enumerate(dec_anch):
        size = (labels_anch == i).sum()
        top3 = ', '.join(f'{repr(t)}: {p:.2f}' for t, p in tokens[:3])
        print(f'    A{i} ({size}): {top3}')

    print(f'\n--- Protocol 3: Damped dynamics ({L} steps) ---')
    h_dyn = simulate_damped_simple(model, xi, seeds, n_steps=L)
    K_dyn, labels_dyn, centers_dyn, sils_dyn = sweep_clusters(h_dyn)
    dec_dyn = decode_centroids(centers_dyn)
    print(f'  K* = {K_dyn}  silhouettes = {sils_dyn}')
    for i, tokens in enumerate(dec_dyn):
        size = (labels_dyn == i).sum()
        top3 = ', '.join(f'{repr(t)}: {p:.2f}' for t, p in tokens[:3])
        print(f'    A{i} ({size}): {top3}')

    attractor_results[prompt_name] = {
        'gd': {'K': K_gd, 'sils': sils_gd, 'decoded': dec_gd,
               'n_converged': n_conv_gd, 'mean_V': float(V_gd.mean()),
               'mean_h_norm': float(np.linalg.norm(h_gd, axis=-1).mean())},
        'anchored': {'K': K_anch, 'sils': sils_anch, 'decoded': dec_anch,
                     'n_converged': n_conv_anch},
        'dyn': {'K': K_dyn, 'sils': sils_dyn, 'decoded': dec_dyn},
    }

attr_path = RUN_DIR / f'{full_tag}_attractor_results.json'
with open(attr_path, 'w') as f:
    json.dump(attractor_results, f, indent=2, default=str)
print(f'\nwrote attractor results: {attr_path}')

## 9. V_θ landscape diagnostics

In [ ]:
v_samples = []
model.eval()
with torch.no_grad():
    for _ in range(10):
        xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        with torch.enable_grad():
            out = model(x, return_trajectory=True)
        traj = out[2]
        h_L = traj[-1].to(device)
        xi = causal_cumulative_mean(h_L)
        V_vals = model.V_theta(xi, h_L).detach().cpu().numpy().ravel()
        v_samples.append(V_vals)

V_all = np.concatenate(v_samples)
print(f'V_theta on real trajectories ({CELL}):')
print(f'  mean   = {V_all.mean():.2f}')
print(f'  std    = {V_all.std():.2f}')
print(f'  min    = {V_all.min():.2f}')
print(f'  max    = {V_all.max():.2f}')
print(f'  range  = {V_all.max() - V_all.min():.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(V_all, bins=100, color='#3a6ea5', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('V_\u03b8(\u03be, h)')
ax.set_ylabel('count')
ax.set_title(f'{CELL} V_\u03b8 distribution ({recipe["desc"]})')
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_v_theta_hist.png', dpi=120)
plt.show()

landscape_stats = {
    'mean': float(V_all.mean()), 'std': float(V_all.std()),
    'min': float(V_all.min()), 'max': float(V_all.max()),
    'range': float(V_all.max() - V_all.min()),
}
with open(RUN_DIR / f'{full_tag}_landscape_stats.json', 'w') as f:
    json.dump(landscape_stats, f, indent=2)

## 10. Cross-cell comparison dashboard

In [ ]:
results = {}
for cell_name in ('P1', 'P2', 'P3', 'P4', 'P5', 'G1', 'G2', 'G3'):
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    last = rows[-1] if rows else None
    best = min(r['val_ppl'] for r in rows) if rows else None
    attr_files = sorted(cell_dir.glob('*_attractor_results.json'))
    attr = None
    if attr_files:
        attr = json.loads(attr_files[-1].read_text())
    ls_files = sorted(cell_dir.glob('*_landscape_stats.json'))
    ls = None
    if ls_files:
        ls = json.loads(ls_files[-1].read_text())
    results[cell_name] = {
        'desc': IMPROVEMENT_RECIPES[cell_name]['desc'],
        'val_ppl': last['val_ppl'] if last else None,
        'best_ppl': best,
        'attractors': attr,
        'landscape': ls,
    }

print(f'{"Cell":<6} {"Description":<45} {"best PPL":>10} {"final PPL":>10} '
      f'{"V range":>8}')
print('-' * 90)

# Reference baselines
print(f'{"":<6} {"FR4 baseline (FockPARF, \u03bb_V=1)":<45} {"190.2":>10} '
      f'{"203.4":>10} {"3.0":>8}')
print(f'{"":<6} {"Attention baseline (8L GPT-2)":<45} {"~150":>10} '
      f'{"-":>10} {"-":>8}')
print(f'{"":<6} {"Hybrid SPLM+Attn (k=4,m=4)":<45} {"133.0":>10} '
      f'{"-":>10} {"-":>8}')
print(f'{"":<6} {"SPLM em_ln (v_hidden=512, \u03b3=0.10)":<45} {"173.6":>10} '
      f'{"-":>10} {"-":>8}')
print(f'{"":<6} {"PARF reg (\u03bb_V=1e-4, PR2)":<45} {"175.6":>10} '
      f'{"-":>10} {"-":>8}')
print('-' * 90)

for cell_name, r in results.items():
    if r is None:
        desc = IMPROVEMENT_RECIPES[cell_name]['desc']
        print(f'{cell_name:<6} {desc:<45} {"\u2014":>10} {"\u2014":>10} '
              f'{"\u2014":>8}    (not run)')
        continue
    desc = r['desc']
    ppl_final = f"{r['val_ppl']:.1f}" if r['val_ppl'] else '\u2014'
    ppl_best = f"{r['best_ppl']:.1f}" if r['best_ppl'] else '\u2014'
    v_range = f"{r['landscape']['range']:.1f}" if r['landscape'] else '\u2014'
    print(f'{cell_name:<6} {desc:<45} {ppl_best:>10} {ppl_final:>10} '
          f'{v_range:>8}')

print(f'\n\u2192  Target: beat attention baseline (~150 PPL) or at minimum '
      f'close the 40-PPL gap from FR4 (190.2).')